In [1]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Basic info
df.head()
df.info()
df.describe()

# Check missing values
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [3]:
import matplotlib.pyplot as plt

# Convert target to numeric
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Example analysis
print(df.groupby('Contract')['Churn'].mean())
print(df.groupby('tenure')['Churn'].mean())
print(df.groupby('MonthlyCharges')['Churn'].mean())

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64
tenure
0     0.000000
1     0.619902
2     0.516807
3     0.470000
4     0.471591
        ...   
68    0.090000
69    0.084211
70    0.092437
71    0.035294
72    0.016575
Name: Churn, Length: 73, dtype: float64
MonthlyCharges
18.25     0.0
18.40     0.0
18.55     0.0
18.70     0.0
18.75     0.0
         ... 
118.20    0.0
118.35    1.0
118.60    0.0
118.65    0.0
118.75    0.0
Name: Churn, Length: 1585, dtype: float64


/tmp/ipykernel_613/4156568047.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [4]:
df = pd.get_dummies(df, drop_first=True)

In [6]:
from sklearn.model_selection import train_test_split

# Clean column names (fix hidden spaces issue)
df.columns = df.columns.str.strip()

# Drop columns safely (only if they exist)
cols_to_drop = ['Churn']
if 'customerID' in df.columns:
    cols_to_drop.append('customerID')

X = df.drop(cols_to_drop, axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [9]:
from sklearn.metrics import classification_report, accuracy_score

print("Logistic Regression:\n")
print(classification_report(y_test, y_pred_lr))

print("Decision Tree:\n")
print(classification_report(y_test, y_pred_dt))

Logistic Regression:

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.68      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409

Decision Tree:

              precision    recall  f1-score   support

           0       0.83      0.93      0.88      1036
           1       0.71      0.46      0.56       373

    accuracy                           0.81      1409
   macro avg       0.77      0.70      0.72      1409
weighted avg       0.80      0.81      0.79      1409



In [10]:
print(y.value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [11]:
import numpy as np

importances = dt.feature_importances_
features = X.columns

feat_imp = pd.Series(importances, index=features)
top_features = feat_imp.sort_values(ascending=False).head(3)

print(top_features)

tenure                         0.448941
InternetService_Fiber optic    0.340557
TotalCharges                   0.045134
dtype: float64


## Business Summary

Our analysis shows that customers with month-to-month contracts, shorter tenure, and higher monthly charges are more likely to churn. This indicates that customers who are not committed long-term tend to leave the service more frequently. The Decision Tree model helped identify the most important factors driving churn, while Logistic Regression provided stable predictive performance.

From a business perspective, offering long-term contracts, discounts for loyal customers, and reducing monthly costs could significantly reduce churn. Targeting high-risk customers early can help improve customer retention and increase overall revenue.
